# Stage 1 — Domain Adaptation with LoRA (non-instructional fine-tuning)

**Base model:** `unsloth/Llama-3.2-1B` (ungated mirror of Meta's Llama 3.2 1B **base** weights)
**Data:** [`qiaojin/PubMedQA`](https://huggingface.co/datasets/qiaojin/PubMedQA) — real PubMed
abstracts, MIT licence
**Hardware:** free-tier Colab T4 · **Runtime:** ~12 minutes

---

## What this notebook does

Continues training a pretrained base model on **raw domain text** — no instructions, no prompts,
no expected answers. Just next-token prediction over ~290k tokens of biomedical abstracts. This is
*domain-adaptive pretraining*, *continued pretraining*, or *non-instructional fine-tuning*.

## Why PubMed abstracts

Biomedical abstracts are a good domain-adaptation target for three reasons:

1. **Genuinely specialised.** A 1B general model is measurably worse at this text than at
   Wikipedia, so there is real headroom for perplexity to fall. Fine-tuning on Wikipedia would
   show almost nothing, because the model already knows it.
2. **Strong structural regularity.** Abstracts follow `BACKGROUND: … METHODS: … RESULTS: …`.
   That pattern is learnable and visibly so.
3. **The same dataset feeds stage 2.** PubMedQA's `pqa_labeled` config carries expert-annotated
   question/answer pairs over *different* articles, so notebook 02 gets an objective accuracy
   metric from the same source. No stitching two unrelated datasets together.

This notebook uses the `pqa_unlabeled` config (61k articles); notebook 02 uses `pqa_labeled`
(1k articles). They are disjoint by construction, and we assert it.

## What to expect

A base model, after this, **still will not answer questions**. It will continue text in the
register of the corpus. That is not a failure — it is what training on raw text asks for.
Instruction-following comes from notebook 02.

**Set your runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.

## 1. Runtime check and dependencies

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "No GPU found — set Runtime > Change runtime type > T4 GPU")

In [ ]:
# Pinned to match requirements-colab.txt. torch is deliberately left alone:
# Colab ships a CUDA-matched build and replacing it is slow and fragile.
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model(). Nothing here uses torchao, so remove it rather than chase a
# compatible build against Colab's torch.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

In [ ]:
import torch, transformers, peft, datasets, accelerate

print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"datasets     {datasets.__version__}")
print(f"accelerate   {accelerate.__version__}")

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU."
gpu = torch.cuda.get_device_name(0)
print(f"\nGPU: {gpu}")

# bfloat16 needs Ampere (sm_80) or later. A T4 is Turing (sm_75), so this notebook uses fp16.
# We check compute capability directly: torch.cuda.is_bf16_supported() has historically
# returned True on Turing, where bf16 is emulated and slow.
CC_MAJOR, CC_MINOR = torch.cuda.get_device_capability()
SUPPORTS_BF16 = CC_MAJOR >= 8
print(f"compute capability: sm_{CC_MAJOR}{CC_MINOR}")
print(f"native bfloat16   : {SUPPORTS_BF16}  ->  using {'bf16' if SUPPORTS_BF16 else 'fp16'}")

# transformers v5 renamed the `torch_dtype` argument of from_pretrained to `dtype`.
# Pick the right one so the notebook survives the fallback pin set too.
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"
print(f"from_pretrained dtype keyword: {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

## 2. Load the corpus straight from Hugging Face

No repo checkout needed — this notebook is self-contained. `scripts/prepare_data.py` in the repo
runs exactly this logic offline if you want to inspect the data without a GPU.

A PubMedQA record looks like this:

```python
{"pubid": 14499029,
 "question": "Is naturopathy as effective as conventional therapy ...?",
 "context": {"contexts": ["Although the use of alternative medicine ...", ...],
             "labels":   ["BACKGROUND", "OBJECTIVE", "DESIGN", ...]},
 "long_answer": "..."}
```

For stage 1 we throw away the questions entirely and keep only `context.contexts` — the raw
abstract prose. **That is what makes this non-instructional.**

In [ ]:
from datasets import load_dataset

# --- constants mirrored in scripts/prepare_data.py and notebook 02 ---
DATASET = "qiaojin/PubMedQA"
SEED = 20260815
N_ABSTRACTS = 1000         # sampled from pqa_unlabeled (61,249 available)
MIN_SECTION_CHARS = 120    # drop abstract sections shorter than this

unlabeled = load_dataset(DATASET, "pqa_unlabeled", split="train")
print(unlabeled)

In [ ]:
sampled = unlabeled.shuffle(seed=SEED).select(range(N_ABSTRACTS))

corpus = []
for row in sampled:
    labels = row["context"].get("labels") or []
    for i, text in enumerate(row["context"]["contexts"]):
        text = " ".join(text.split())            # collapse whitespace
        if len(text) < MIN_SECTION_CHARS:        # drop fragments
            continue
        corpus.append({
            "text": text,
            "pubid": row["pubid"],
            "section": (labels[i] if i < len(labels) else "").strip() or "UNLABELLED",
        })

CORPUS_PUBIDS = {r["pubid"] for r in corpus}     # notebook 02 checks against this
print(f"{len(corpus):,} abstract sections from {len(CORPUS_PUBIDS):,} articles")
print(f"{sum(len(r['text'].split()) for r in corpus):,} words\n")
print("--- a sample section ---")
print(f"[{corpus[0]['section']}] {corpus[0]['text'][:400]}")

## 3. What's actually in the corpus

Worth a minute before training on it.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

words = [len(r["text"].split()) for r in corpus]
print(f"words per section: min={min(words)} median={sorted(words)[len(words)//2]} max={max(words)}")
print(f"sections per abstract: {len(corpus) / len(CORPUS_PUBIDS):.1f}\n")

print("most common section headings:")
for label, n in Counter(r["section"] for r in corpus).most_common(12):
    print(f"  {label:28s} {n:5d}")

plt.figure(figsize=(9, 3))
plt.hist(words, bins=50, color="#4C72B0", edgecolor="white")
plt.xlabel("words per abstract section"); plt.ylabel("count")
plt.title("PubMedQA corpus — section lengths"); plt.tight_layout(); plt.show()

## 4. The tokenizer — two gotchas

**Gotcha 1: the pad token.** Most tutorials do `tokenizer.pad_token = tokenizer.eos_token`,
because Llama 2 shipped without a pad token. Llama 3.2 has a real one
(`<|finetune_right_pad_id|>`, id `128004`). Aliasing EOS as PAD when a real pad token exists makes
padding and end-of-sequence indistinguishable — and EOS is exactly the token you need the model to
learn in notebook 02.

**Gotcha 2: padding side.** Llama 3.2's tokenizer config sets `padding_side="left"`, correct for
*batched generation* and wrong for *training*. We flip it to `"right"`.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "unsloth/Llama-3.2-1B"   # ungated mirror of meta-llama/Llama-3.2-1B (base, not Instruct)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"vocab size    : {len(tokenizer):,}")
print(f"eos           : {tokenizer.eos_token!r} -> {tokenizer.eos_token_id}")
print(f"pad           : {tokenizer.pad_token!r} -> {tokenizer.pad_token_id}")
print(f"padding_side  : {tokenizer.padding_side}  (config default)")

assert tokenizer.pad_token_id is not None, "No pad token — you would need to alias EOS here"
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"

tokenizer.padding_side = "right"
print(f"padding_side  : {tokenizer.padding_side}  (flipped for training)")

# Biomedical text tokenizes worse than general English — technical terms fragment into
# more subwords. Worth knowing, because it inflates every token budget downstream.
sample = " ".join(r["text"] for r in corpus[:200])
ratio = len(tokenizer(sample, add_special_tokens=False)["input_ids"]) / len(sample.split())
print(f"\ntokens per word on this corpus: {ratio:.2f}  (general English is ~1.3)")

## 5. Turning text into training blocks

Two ways to do this, and the difference is not cosmetic.

**The common tutorial approach** tokenizes each passage with `padding="max_length"` and then sets
`labels = input_ids.copy()`. Two problems: most of every sequence is padding, and because the
labels are a straight copy, **the model is trained to predict pad tokens**. The loss falls
smoothly — the model gets very good at predicting padding — while learning much less than the
curve suggests.

**Packing** concatenates the corpus into one token stream and slices it into fixed-length blocks.
Every supervised token is real text. This is how language models are actually pretrained, and
continued pretraining should match it.

Let's measure the difference rather than assert it.

In [ ]:
BLOCK_SIZE = 512

# --- Approach A: pad each section to max_length (what many tutorials do) ---
naive = tokenizer([r["text"] for r in corpus],
                  truncation=True, padding="max_length", max_length=BLOCK_SIZE)
naive_total = sum(len(ids) for ids in naive["input_ids"])
naive_real = sum(sum(mask) for mask in naive["attention_mask"])

print("A. pad-to-max_length")
print(f"   sequences       : {len(naive['input_ids']):,}")
print(f"   token slots     : {naive_total:,}")
print(f"   real tokens     : {naive_real:,}")
print(f"   useful fraction : {naive_real / naive_total:.1%}   <- the rest is padding")

In [ ]:
# --- Approach B: concatenate and chunk (what we use) ---
# Sections of one abstract flow together; one EOS between abstracts so the model learns
# where a document ends.
by_article = {}
for r in corpus:
    by_article.setdefault(r["pubid"], []).append(f"{r['section']}: {r['text']}")

stream = []
for pubid in sorted(by_article):
    stream.extend(tokenizer("\n\n".join(by_article[pubid]), add_special_tokens=False)["input_ids"])
    stream.append(tokenizer.eos_token_id)

n_blocks = len(stream) // BLOCK_SIZE            # drop the ragged tail
blocks = [stream[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE] for i in range(n_blocks)]

print("B. concatenate and chunk")
print(f"   stream length   : {len(stream):,} tokens")
print(f"   blocks          : {len(blocks)} x {BLOCK_SIZE}")
print(f"   useful fraction : 100.0%   <- no padding at all")
print(f"   discarded tail  : {len(stream) % BLOCK_SIZE} tokens")
print(f"\nApproach A would have spent {1 - naive_real / naive_total:.0%} of the training budget "
      f"learning to predict padding.")

In [ ]:
# Sanity check: a block should decode to readable abstract prose.
print(tokenizer.decode(blocks[7][:300]))

## 6. Train / eval split

We hold out 10% of the **blocks**. This measures *in-domain fit on held-out passages*. Because
blocks are packed from many articles, a held-out block may contain text from an article that also
contributed to a training block.

For a strict generalisation measure you would split at **article** level before packing. That is
the more rigorous choice; in-domain fit is the right target here, since the goal of domain
adaptation is for the model to be less surprised by text of this kind. Stage 2's evaluation, which
is the one that carries a real metric, is fully disjoint at article level.

In [ ]:
import random
from datasets import Dataset

random.seed(42)
indices = list(range(len(blocks)))
random.shuffle(indices)

n_eval = max(8, int(len(blocks) * 0.10))
eval_idx, train_idx = indices[:n_eval], indices[n_eval:]


def to_dataset(idx):
    rows = [blocks[i] for i in idx]
    return Dataset.from_dict({
        "input_ids": rows,
        "attention_mask": [[1] * BLOCK_SIZE for _ in rows],
        # Causal LM: the labels ARE the inputs. The model shifts them internally, so
        # position i predicts token i+1. No manual shifting needed.
        "labels": [list(r) for r in rows],
    })


train_ds, eval_ds = to_dataset(train_idx), to_dataset(eval_idx)
print(f"train: {len(train_ds)} blocks  ({len(train_ds) * BLOCK_SIZE:,} tokens)")
print(f"eval : {len(eval_ds)} blocks  ({len(eval_ds) * BLOCK_SIZE:,} tokens)")

## 7. Baseline — measure BEFORE you train

The step most tutorials omit, and omitting it makes every later number meaningless. A reported
perplexity of 8.4 after training tells you nothing unless you know what it was before.

In [ ]:
from transformers import AutoModelForCausalLM

DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE})
model.to("cuda")
model.config.pad_token_id = tokenizer.pad_token_id

print(f"{MODEL_ID}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters, {DTYPE}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
import math


@torch.no_grad()
def perplexity(model, dataset, batch_size=4):
    """Token-weighted perplexity over a tokenized dataset.

    HF returns the *mean* loss per batch, so batches must be re-weighted by how many
    positions actually contributed before they can be averaged together. Labels set to
    -100 are ignored by the loss, and the model shifts internally (position i predicts
    i+1), so the contributing positions are exactly `labels[:, 1:] != -100`.
    """
    model.eval()
    total_nll, total_tokens = 0.0, 0
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i + batch_size]
        ids = torch.tensor(batch["input_ids"], device="cuda")
        mask = torch.tensor(batch["attention_mask"], device="cuda")
        labels = torch.tensor(batch["labels"], device="cuda")
        out = model(input_ids=ids, attention_mask=mask, labels=labels)
        n = int((labels[:, 1:] != -100).sum().item())
        total_nll += out.loss.item() * n
        total_tokens += n
    return math.exp(total_nll / total_tokens)


baseline_ppl = perplexity(model, eval_ds)
print(f"BASELINE perplexity on held-out PubMed abstracts: {baseline_ppl:.3f}")

In [ ]:
# Probe prompts in the corpus's own idiom. A domain-adapted model should continue these
# as structured abstract prose; the base model will drift into generic web text.
PROBES = [
    "BACKGROUND: Metformin is the most widely prescribed oral antihyperglycemic agent, but",
    "METHODS: We conducted a randomized controlled trial in which patients were assigned to",
    "RESULTS: Compared with the control group, the intervention group showed a significant",
    "CONCLUSIONS: These findings suggest that",
]


@torch.no_grad()
def complete(model, prompt, max_new_tokens=70):
    model.eval()
    model.config.use_cache = True
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


baseline_completions = {p: complete(model, p) for p in PROBES}
for p, c in baseline_completions.items():
    print(f"\nPROMPT : {p}")
    print(f"BASE   : {c}")

## 8. LoRA configuration

LoRA freezes the pretrained weight $W$ and learns a low-rank update beside it:

$$W' = W + \frac{\alpha}{r} BA \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k}$$

Only $A$ and $B$ train. With $r \ll d$ that is a fraction of a percent of the parameters.

**`target_modules`** — the common default is `["q_proj", "v_proj"]`, which suits *behavioural*
change. We target all seven projections, including the MLP (`gate_proj`, `up_proj`, `down_proj`),
because factual and lexical knowledge lives disproportionately in the feed-forward layers, and
absorbing a new vocabulary is exactly what domain adaptation is for. Teams reporting disappointing
domain-adaptation results have usually adapted attention only.

**`r` and `lora_alpha`** — $r$ sets capacity; $\alpha/r$ scales how hard the update pushes.
$\alpha = 2r$ is a common, sane default. $r=16$ suits domain adaptation on a corpus this size.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### One more gotcha: fp16 weights + fp16 optimizer

If you load the model in fp16 and train with `fp16=True`, the trainable LoRA parameters are also
fp16 — and PyTorch's gradient scaler raises `ValueError: Attempting to unscale FP16 gradients.`

The fix is to keep the *base* weights in fp16 (that's where the memory is) and upcast only the
**trainable** parameters to fp32. This is exactly what `prepare_model_for_kbit_training` does for
QLoRA; doing it explicitly makes it visible.

In [ ]:
upcast = 0
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
        upcast += 1
print(f"upcast {upcast} trainable tensors to fp32 (base weights stay in {DTYPE})")

# Turn this on for larger models or longer sequences: it trades ~30% speed for a large
# activation-memory saving. A 1B model at 512 tokens doesn't need it.
USE_GRADIENT_CHECKPOINTING = False
if USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()   # required, or PEFT gradients never reach the adapter

model.config.use_cache = False   # incompatible with training; we turn it back on to generate
print(f"gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}")

## 9. Train

~510 training blocks at an effective batch size of 4 gives ~128 optimizer steps per epoch. Two
epochs is enough here — with ~290k tokens the model has plenty to learn from and doesn't need to
see it five times. Watch the **eval** loss: if it turns up while train loss keeps falling, that's
overfitting and you should cut the epochs.

`Trainer` gets a `data_collator` but **no tokenizer**. It doesn't need one — the data is already
tokenized to fixed length — and skipping it avoids the `tokenizer=` → `processing_class=` rename
between transformers 4.x and 5.x.

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

OUT_DIR = "/content/outputs/stage1-domain-lora"

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    logging_steps=10,
    eval_strategy="epoch",     # renamed from `evaluation_strategy` in transformers 4.41
    save_strategy="no",        # we save the adapter ourselves at the end
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=default_data_collator,
)

print(f"optimizer steps: ~{len(train_ds) // args.per_device_train_batch_size * args.num_train_epochs}")
trainer.train()

## 10. Loss curves

In [ ]:
history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*train_pts), label="train loss", color="#4C72B0", alpha=0.8)
if eval_pts:
    plt.plot(*zip(*eval_pts), label="eval loss", color="#C44E52", marker="o")
    best = min(eval_pts, key=lambda p: p[1])
    plt.annotate(f"best eval {best[1]:.3f}", best, textcoords="offset points",
                 xytext=(10, 15), fontsize=9)
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Stage 1: domain adaptation on PubMed abstracts")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

if eval_pts and eval_pts[-1][1] > min(p[1] for p in eval_pts) * 1.02:
    print("Eval loss rose after its minimum -> overfitting. Reduce num_train_epochs.")

## 11. Did it work? Perplexity, before vs after

In [ ]:
adapted_ppl = perplexity(model, eval_ds)
delta = (baseline_ppl - adapted_ppl) / baseline_ppl

print(f"{'':26s}{'perplexity':>12s}")
print(f"{'-' * 38}")
print(f"{'base model':26s}{baseline_ppl:>12.3f}")
print(f"{'+ domain LoRA':26s}{adapted_ppl:>12.3f}")
print(f"{'-' * 38}")
print(f"{'relative change':26s}{-delta:>11.1%}")

if adapted_ppl < baseline_ppl:
    print(f"\nPerplexity fell {delta:.1%}: the model is less surprised by held-out abstracts.")
else:
    print("\nPerplexity did not improve. Check the learning rate, or train for more epochs.")

### A perplexity check that isn't circular

Perplexity fell on biomedical text — but did the model get worse at everything else? A domain
fine-tune that improves its target while degrading general capability has **traded**, not gained.
Here's an honest check on clearly out-of-domain prose.

In [ ]:
OUT_OF_DOMAIN = [
    "In 1687 Isaac Newton published the Principia, setting out the laws of motion and universal "
    "gravitation. The work unified terrestrial and celestial mechanics under a single set of "
    "principles and remained the dominant framework in physics for over two centuries, until "
    "relativity and quantum mechanics revealed its limits at extreme scales.",
    "A good risotto depends less on the recipe than on attention. The rice is toasted in fat "
    "until the grains turn translucent at the edges, then stock is added a ladle at a time, "
    "each addition absorbed before the next. The constant stirring releases starch, which is "
    "what gives the finished dish its texture rather than any cream.",
    "The doctrine of consideration requires that a promise be supported by something of value "
    "given in return before a court will enforce it as a contract. Nominal consideration is "
    "generally sufficient, since courts assess the existence of consideration rather than its "
    "adequacy, leaving the parties to judge the value of their own bargain.",
]

ood = tokenizer(OUT_OF_DOMAIN, truncation=True, max_length=BLOCK_SIZE, padding="longest")
# Practising what section 5 preached: pad positions are masked out of the loss with -100,
# so perplexity is computed over real tokens only.
ood_ds = Dataset.from_dict({
    "input_ids": ood["input_ids"],
    "attention_mask": ood["attention_mask"],
    "labels": [[tok if m else -100 for tok, m in zip(ids, mask)]
               for ids, mask in zip(ood["input_ids"], ood["attention_mask"])],
})

with model.disable_adapter():
    ood_base = perplexity(model, ood_ds)
ood_adapted = perplexity(model, ood_ds)

print(f"out-of-domain perplexity   base: {ood_base:.3f}   adapted: {ood_adapted:.3f}")
print(f"change: {(ood_adapted - ood_base) / ood_base:+.1%}")
print(f"in-domain change for comparison: {-delta:+.1%}")
print("\nSome regression here is the normal cost of specialising. What you want is a much larger "
      "in-domain gain than out-of-domain loss.")

## 12. Qualitative check

`disable_adapter()` is a PEFT context manager that switches the LoRA off in place, so we get an
exact base-model comparison **without loading a second copy of the model**. Same weights, same
prompt, adapter on versus off.

In [ ]:
model.config.use_cache = True

for prompt in PROBES:
    with model.disable_adapter():
        before = complete(model, prompt)
    after = complete(model, prompt)
    print("=" * 100)
    print(f"PROMPT   {prompt}")
    print(f"\nBASE     {before}")
    print(f"\nADAPTED  {after}")
print("=" * 100)

**What to look for.** The adapted model should produce clinical-trial prose: effect sizes,
confidence intervals, `P` values, and the next section header (`CONCLUSIONS:` after `RESULTS:`).
The base model tends to drift into generic explanatory text or start listing unrelated headings.

It should **not** be answering questions politely — it is a base model that has read a lot of
abstracts, so it continues text.

In [ ]:
# Demonstrating the point: a base model does not follow instructions.
print(complete(model, "What is metformin used to treat?", max_new_tokens=80))

### Did it memorise, or generalise?

A model that reproduces training text verbatim has stopped generalising. Feed it the opening of a
*training* abstract and measure how much of the real continuation it reproduces word for word.

In [ ]:
def overlap_fraction(generated: str, reference: str, n: int = 8) -> float:
    """Fraction of the reference's n-gram set that also appears in the generation."""
    g, r = generated.lower().split(), reference.lower().split()
    if len(r) < n:
        return 0.0
    gset = {tuple(g[i:i + n]) for i in range(len(g) - n + 1)}
    rset = {tuple(r[i:i + n]) for i in range(len(r) - n + 1)}
    return len(gset & rset) / len(rset) if rset else 0.0


train_pubids = {r["pubid"] for r in corpus}
scores = []
for r in corpus[:40]:
    words = r["text"].split()
    if len(words) < 90:
        continue
    prompt, reference = " ".join(words[:25]), " ".join(words[25:])
    scores.append(overlap_fraction(complete(model, prompt, max_new_tokens=90), reference))
    if len(scores) >= 12:
        break

print(f"mean 8-gram overlap with the true continuation: {sum(scores) / len(scores):.1%}")
print("Low single digits is healthy. Anything above ~30% means the model is reciting rather")
print("than generalising — cut the epochs or add data.")

## 13. Save the adapter

We save **only the adapter** (~45 MB), not a merged model (~2.5 GB). The adapter plus the base
model id is everything notebook 02 needs, and it's what you'd version in a real registry.

In [ ]:
from pathlib import Path

ADAPTER_DIR = Path(OUT_DIR)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Saved to {ADAPTER_DIR}  ({size_mb:.1f} MB)")
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
# Persist to Drive so notebook 02 can pick it up. Colab wipes /content on disconnect.
import shutil

DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/finetuning-demo/stage1-domain-lora")
    DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_DIR.exists():
        shutil.rmtree(DRIVE_DIR)
    shutil.copytree(ADAPTER_DIR, DRIVE_DIR)
    print(f"\nCopied to {DRIVE_DIR}")
    print("\n>>> Notebook 02 will look for exactly this path. <<<")
except ImportError:
    print("Not running in Colab — the adapter is on local disk at:")
    print(f"  {ADAPTER_DIR}")

In [ ]:
# Fallback if you'd rather not use Drive: download a zip.
# shutil.make_archive("/content/stage1-domain-lora", "zip", ADAPTER_DIR)
# from google.colab import files; files.download("/content/stage1-domain-lora.zip")

## 14. What just happened, and what didn't

**What happened.** A base model read ~290k tokens of real PubMed abstracts under a plain
next-token objective. ~11M LoRA parameters (0.9% of the model) absorbed the shift, and perplexity
on held-out abstracts fell.

**What did not happen.** It did not learn to follow instructions, answer questions, stop at a
sensible point, or decline when it doesn't know. None of those are in the training signal. Raw
text teaches continuation.

### Honest limitations

- **290k tokens is still small.** Real domain-adaptive pretraining uses 10⁸–10¹⁰ tokens.
  BioMedLM and PubMedBERT trained on the whole of PubMed. Expect register and vocabulary
  adoption plus a measurable perplexity gain — not new clinical capability.
- **The eval blocks are packed from the same article pool as training**, so this is in-domain fit
  rather than generalisation to unseen articles. Notebook 02's metric is fully disjoint.
- **Perplexity is not usefulness.** It says the model finds this text less surprising. It says
  nothing about whether the model's biomedical claims are correct.
- **Nothing here is medical advice**, and a 1B model fine-tuned on abstracts is not a clinical
  tool. This is a training-mechanics exercise.

### Next

`02_instruction_finetuning_lora.ipynb` merges this adapter into the base weights and trains a
second LoRA on PubMedQA's expert-annotated question/answer pairs — with a real accuracy metric to
score it against.